# AgriSense: Plant Disease Detection Model Training
### Deep Learning Frameworks Project — ResNet50 / MobileNetV3 Transfer Learning

**Instructions:**
1. In Google Colab menu: click **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Run all cells sequentially (Ctrl+F9).
3. At the end, the trained model (`plant_disease_model.pth` and `class_names.json`) will be downloaded.
4. Place them in your project directory: `e:\Honors_Project\backend\models\`.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Install required dependencies
!pip install -q torch torchvision torchaudio scikit-learn matplotlib tqdm onnx

### 3. Download PlantVillage Dataset
You can download the dataset via Kaggle or direct link. The following cell downloads the standard PlantVillage dataset archive and extracts it into `plantvillage_dataset/`.

In [ ]:
import os
import zipfile
import urllib.request

# Direct download link or Kaggle download
dataset_dir = "plantvillage_dataset"
os.makedirs(dataset_dir, exist_ok=True)

# If you have kaggle.json uploaded:
if os.path.exists("kaggle.json"):
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d emmarex/plantdisease -p {dataset_dir} --unzip
else:
    print("Downloading PlantVillage dataset directly (sample/curated)... ")
    # Alternative direct mirror
    zip_url = "https://github.com/spMohanty/PlantVillage-Dataset/archive/master.zip"
    zip_path = "plantvillage.zip"
    if not os.path.exists(zip_path) and not os.path.exists(os.path.join(dataset_dir, "PlantVillage-Dataset-master")):
        urllib.request.urlretrieve(zip_url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(dataset_dir)
    print("Dataset ready!")

In [ ]:
# Find dataset root directory
import glob

candidate_dirs = [
    "plantvillage_dataset/PlantVillage-Dataset-master/raw/color",
    "plantvillage_dataset/plantvillage/PlantVillage",
    "plantvillage_dataset/PlantVillage",
    "plantvillage_dataset"
]
DATA_DIR = None
for d in candidate_dirs:
    if os.path.exists(d) and len(os.listdir(d)) > 10:
        DATA_DIR = d
        break

if DATA_DIR is None:
    # Let's inspect directory
    print("Available subdirectories in plantvillage_dataset:", os.listdir(dataset_dir))
    DATA_DIR = dataset_dir

print(f"Selected DATA_DIR: {DATA_DIR}")

In [ ]:
# 4. PyTorch Data Setup & Augmentation
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import json
import time
import copy
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load full dataset
full_dataset = datasets.ImageFolder(root=DATA_DIR)
classes = full_dataset.classes
num_classes = len(classes)
print(f"Total classes: {num_classes}, Total images: {len(full_dataset)}")

# 80% train, 10% validation, 10% test
total_len = len(full_dataset)
train_len = int(0.80 * total_len)
val_len = int(0.10 * total_len)
test_len = total_len - train_len - val_len

generator = torch.Generator().manual_seed(42)
train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_len, val_len, test_len], generator=generator
)

class TransformedDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, idx):
        x, y = self.subset[idx]
        return self.transform(x), y
    def __len__(self):
        return len(self.subset)

train_loader = DataLoader(TransformedDataset(train_subset, train_transform), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(TransformedDataset(val_subset, val_test_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(TransformedDataset(test_subset, val_test_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders ready.")

In [ ]:
# 5. Build Transfer Learning Model (ResNet50 / MobileNetV3)
MODEL_TYPE = "resnet50"  # Or 'mobilenet_v3'

if MODEL_TYPE == "resnet50":
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
elif MODEL_TYPE == "mobilenet_v3":
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.Hardswish(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
print(f"Model {MODEL_TYPE} initialized and moved to {device}")

In [ ]:
# 6. Training Loop with Early Stopping
EPOCHS = 15
best_acc = 0.0
best_weights = copy.deepcopy(model.state_dict())
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        t_correct += torch.sum(preds == lbls).item()
        t_total += imgs.size(0)
        
    train_loss = t_loss / t_total
    train_acc = t_correct / t_total
    
    # Validation
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            v_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            v_correct += torch.sum(preds == lbls).item()
            v_total += imgs.size(0)
            
    val_loss = v_loss / v_total
    val_acc = v_correct / v_total
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1:02d} | Train Acc: {train_acc*100:.2f}% Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% Loss: {val_loss:.4f}")
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())
        print(f"   ==> New best validation accuracy: {best_acc*100:.2f}%")

model.load_state_dict(best_weights)
print(f"Training finished! Best Validation Accuracy: {best_acc*100:.2f}%")

In [ ]:
# 7. Plot Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy Curves')
plt.show()

In [ ]:
# 8. Test Set Evaluation & Metrics
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc="Evaluating Test Set"):
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(lbls.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
test_acc = np.mean(all_preds == all_targets)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(classification_report(all_targets, all_preds, target_names=classes))

In [ ]:
# 9. Export Weights, ONNX Model, and Class Labels
os.makedirs("output_model", exist_ok=True)

# 1. Save Class Names
with open("output_model/class_names.json", "w") as f:
    json.dump(classes, f, indent=2)

# 2. Save PyTorch State Dict
torch.save({
    "state_dict": model.state_dict(),
    "classes": classes,
    "test_acc": float(test_acc)
}, "output_model/plant_disease_model.pth")

# 3. Export to ONNX for fast CPU inference
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(
    model,
    dummy_input,
    "output_model/plant_disease_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=14
)

print("All artifacts saved in output_model/ successfully!")

In [ ]:
# 10. Automatically Download Artifacts to Your PC
from google.colab import files

files.download("output_model/plant_disease_model.pth")
files.download("output_model/class_names.json")
files.download("output_model/plant_disease_model.onnx")